<a href="https://colab.research.google.com/github/upen0303/sdSCA-multi-robot-path-planning/blob/main/notebooks/sdSCA_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Cell 1: Clone GitHub Repository ───────────────────────
import os

# Your GitHub repo URL
GITHUB_USERNAME = "your_username"          # ← change this
REPO_NAME       = "sdSCA-multi-robot-path-planning"
REPO_URL        = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

# Clone repository
if os.path.exists(REPO_NAME):
    print("Repo already exists — pulling latest...")
    os.chdir(REPO_NAME)
    os.system("git pull origin main")
else:
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL}")
    os.chdir(REPO_NAME)

print(f"\nCurrent directory: {os.getcwd()}")
print("Files:", os.listdir("."))

In [ ]:
# ── Cell 2: Install Required Libraries ────────────────────
import subprocess

packages = [
    "numpy",
    "matplotlib",
    "scipy",
    "pandas",
    "seaborn",
    "tqdm",
]

for pkg in packages:
    subprocess.run(
        ["pip", "install", pkg, "-q"],
        capture_output=True
    )
    print(f"✅ {pkg} installed")

print("\nAll libraries ready!")

In [ ]:
# ── Cell 3: Verify Everything Works ───────────────────────
import sys
sys.path.append(".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from algorithms.sca     import SCA
from algorithms.sdsca   import sdSCA
from algorithms.qlsdsca import qlsdSCA
from path_planning.scenarios import get_scenario

print("=" * 50)
print("  Environment Verification")
print("=" * 50)
print(f"✅ NumPy    : {np.__version__}")
print(f"✅ Pandas   : {pd.__version__}")
print(f"✅ SCA      : imported")
print(f"✅ sdSCA    : imported")
print(f"✅ qlsdSCA  : imported")
print(f"✅ Scenarios: imported")

# Quick algorithm test
def sphere(x):
    return np.sum(x**2)

sca = SCA(10, 100, 5, -10, 10)
_, best, _ = sca.optimize(sphere)
print(f"\nQuick SCA test: {best:.6f} (should be ~0)")
print("\n✅ Everything working correctly!")

In [ ]:
# ── Cell 4: Experiment Configuration ──────────────────────

# !! IMPORTANT !!
# This is the FULL 30-run experiment
# Estimated time: 6-8 hours on Colab
# Make sure Colab stays open!

CONFIG = {
    "NUM_RUNS"       : 30,     # paper standard
    "MAX_STEPS"      : {
        1: 300,
        2: 400,
        3: 800,
    },
    "OPT_ITERATIONS" : 100,
    "POPULATION"     : 30,
    "ALGORITHMS"     : ["SCA", "sdSCA", "qlsdSCA"],
    "SCENARIOS"      : [1, 2, 3],
}

print("=" * 50)
print("  Experiment Configuration")
print("=" * 50)
for key, val in CONFIG.items():
    print(f"  {key:<20}: {val}")
print("=" * 50)
print("\n⚠️  Estimated time: 6-8 hours")
print("⚠️  Keep this tab open!")

In [ ]:
# ── Cell 5: Helper Functions ───────────────────────────────
import time
import os

def create_algorithm(name, dim, lb, ub, config):
    """Create algorithm by name"""

    PS   = config["POPULATION"]
    ITER = config["OPT_ITERATIONS"]

    if name == "SCA":
        return SCA(PS, ITER, dim, lb, ub, a=2)

    elif name == "sdSCA":
        return sdSCA(
            PS, ITER, dim, lb, ub,
            a=2, F=0.8, CR=0.95
        )

    elif name == "qlsdSCA":
        return qlsdSCA(
            PS, ITER, dim, lb, ub,
            a       = 2,
            F       = 0.8,
            CR      = 0.95,
            alpha   = 0.3,
            gamma   = 0.95,
            epsilon = 1.0
        )

def run_single(scenario_num, algo_name, config,
               verbose=False):
    """Run one simulation"""

    env           = get_scenario(scenario_num)
    env.max_steps = config["MAX_STEPS"][scenario_num]

    NR = len(env.robots)
    D  = NR * 2
    lb = np.tile([1.0, 0.0],     NR)
    ub = np.tile([1.5, 2*np.pi], NR)

    algo = create_algorithm(algo_name, D, lb, ub, config)
    env.set_algorithm(algo)

    results = env.run(verbose=verbose)

    results["steps_per_robot"]     = [
        r.steps_taken    for r in env.robots
    ]
    results["distances_per_robot"] = [
        r.total_distance for r in env.robots
    ]
    results["all_reached_goal"]    = all(
        r.reached_goal   for r in env.robots
    )

    return results

def run_multiple(scenario_num, algo_name, config):
    """Run multiple times and collect results"""

    num_runs = config["NUM_RUNS"]
    print(f"\n  {algo_name} — Scenario {scenario_num} "
          f"({num_runs} runs)")

    all_apde    = []
    all_augd    = []
    all_fitness = []
    all_aet     = []
    all_steps   = []
    all_dist    = []
    all_steps_r = []
    all_dist_r  = []

    for run in range(1, num_runs + 1):

        t0      = time.time()
        results = run_single(
            scenario_num, algo_name, config
        )
        elapsed = time.time() - t0

        all_apde.append(results["APDE"])
        all_augd.append(results["AUGD"])
        all_fitness.append(results["total_fitness"])
        all_aet.append(results["AET"])
        all_steps.append(results["total_steps"])
        all_dist.append(results["total_distance"])
        all_steps_r.append(results["steps_per_robot"])
        all_dist_r.append(results["distances_per_robot"])

        reached = "✅" if results["all_reached_goal"] \
                       else "❌"
        print(f"    Run {run:2d}/{num_runs} | "
              f"Steps={results['total_steps']:5d} | "
              f"APDE={results['APDE']:8.2f} | "
              f"Time={elapsed:.0f}s | {reached}")

    NR = len(all_steps_r[0])

    return {
        "algorithm"            : algo_name,
        "scenario"             : scenario_num,
        "num_runs"             : num_runs,
        "avg_APDE"             : np.mean(all_apde),
        "std_APDE"             : np.std(all_apde),
        "avg_AUGD"             : np.mean(all_augd),
        "std_AUGD"             : np.std(all_augd),
        "avg_total_fitness"    : np.mean(all_fitness),
        "std_fitness"          : np.std(all_fitness),
        "avg_AET"              : np.mean(all_aet),
        "avg_total_steps"      : np.mean(all_steps),
        "avg_total_distance"   : np.mean(all_dist),
        "avg_steps_per_robot"  : [
            np.mean([r[i] for r in all_steps_r])
            for i in range(NR)
        ],
        "avg_dist_per_robot"   : [
            np.mean([r[i] for r in all_dist_r])
            for i in range(NR)
        ],
        "raw_apde"             : all_apde,
        "raw_augd"             : all_augd,
        "raw_fitness"          : all_fitness,
        "raw_aet"              : all_aet,
    }

print("✅ Helper functions ready!")

In [ ]:
# ── Cell 6: Run Scenario 1 ─────────────────────────────────
# Run this cell first — takes ~2 hours

print("=" * 60)
print("  SCENARIO 1 — 6 robots, 100×100 cm")
print("=" * 60)

scenario1_results = {}

for algo in CONFIG["ALGORITHMS"]:
    results = run_multiple(1, algo, CONFIG)
    scenario1_results[algo] = results

    # Save immediately after each algorithm
    os.makedirs("results/scenarios", exist_ok=True)
    df = pd.DataFrame({
        "run"          : range(1, CONFIG["NUM_RUNS"]+1),
        "APDE"         : results["raw_apde"],
        "AUGD"         : results["raw_augd"],
        "total_fitness": results["raw_fitness"],
        "AET"          : results["raw_aet"],
    })
    path = f"results/scenarios/scenario1_{algo}.csv"
    df.to_csv(path, index=False)
    print(f"  ✅ Saved: {path}")

print("\n✅ Scenario 1 complete!")

In [ ]:
# ── Cell 7: Run Scenario 2 ─────────────────────────────────
# ~2 hours

print("=" * 60)
print("  SCENARIO 2 — 7 robots, 100×100 cm")
print("=" * 60)

scenario2_results = {}

for algo in CONFIG["ALGORITHMS"]:
    results = run_multiple(2, algo, CONFIG)
    scenario2_results[algo] = results

    df = pd.DataFrame({
        "run"          : range(1, CONFIG["NUM_RUNS"]+1),
        "APDE"         : results["raw_apde"],
        "AUGD"         : results["raw_augd"],
        "total_fitness": results["raw_fitness"],
        "AET"          : results["raw_aet"],
    })
    path = f"results/scenarios/scenario2_{algo}.csv"
    df.to_csv(path, index=False)
    print(f"  ✅ Saved: {path}")

print("\n✅ Scenario 2 complete!")

In [ ]:
# ── Cell 8: Run Scenario 3 ─────────────────────────────────
# ~4 hours — longest scenario

print("=" * 60)
print("  SCENARIO 3 — 12 robots, 200×200 cm")
print("=" * 60)

scenario3_results = {}

for algo in CONFIG["ALGORITHMS"]:
    results = run_multiple(3, algo, CONFIG)
    scenario3_results[algo] = results

    df = pd.DataFrame({
        "run"          : range(1, CONFIG["NUM_RUNS"]+1),
        "APDE"         : results["raw_apde"],
        "AUGD"         : results["raw_augd"],
        "total_fitness": results["raw_fitness"],
        "AET"          : results["raw_aet"],
    })
    path = f"results/scenarios/scenario3_{algo}.csv"
    df.to_csv(path, index=False)
    print(f"  ✅ Saved: {path}")

print("\n✅ Scenario 3 complete!")

In [ ]:
# ── Cell 9: Push Results to GitHub ────────────────────────
import subprocess

# Configure git
subprocess.run([
    "git", "config", "--global",
    "user.name", "Your Name"      # ← change this
])
subprocess.run([
    "git", "config", "--global",
    "user.email", "your@email.com" # ← change this
])

# Add and commit results
subprocess.run(["git", "add", "results/"])
subprocess.run([
    "git", "commit", "-m",
    "results: 30-run experiments SCA vs sdSCA vs qlsdSCA"
])

# Push — you will need to enter GitHub token
print("\n⚠️  For push, you need GitHub Personal Access Token")
print("Go to: GitHub → Settings → Developer settings")
print("→ Personal access tokens → Generate new token")
print("→ Select 'repo' scope → Generate")
print("\nThen run this in a new cell:")
print("!git push https://TOKEN@github.com/"
      "USERNAME/REPO.git main")

In [ ]:
# ── Cell 10: Final Results Summary ────────────────────────
from analysis.statistical_tests import (
    analyze_scenario,
    print_summary_table
)

all_scenario_results = {}

for num in [1, 2, 3]:
    results = analyze_scenario(num)
    all_scenario_results[num] = results

print_summary_table(all_scenario_results)
print("\n✅ Analysis complete — ready for paper!")